# Predicting Type 2 Diabetes Risk Using XGBoost
---

For this project, the dataset I am using is the Pima Indians Diabetes Dataset — a real, publicly available dataset from the National Institute of Diabetes and Digestive and Kidney Diseases. It contains clinical measurements for 768 female patients of Pima Indian heritage.

The algorithm I am using is XGBoost (Extreme Gradient Boosting), which is one of the most commonly used and well performing models in published T2DM prediction research.

---
## Background: What is XGBoost?

It works by building many small decision trees one after another. Each new tree focuses on correcting the mistakes made by the previous trees.

XGBoost is a strong choice for this project because:
- It handles clinical tabular data (numbers in rows and columns) very well
- It produces a probability score — not just yes/no — which is more useful for risk assessment
- It has a built-in feature importance tool that shows which variables matter most

---
## About the Dataset

The Pima Indians Diabetes Dataset contains **8 clinical features** and 1 outcome column:

| Column | Description |
|---|---|
| Pregnancies | Number of times pregnant |
| Glucose | Plasma glucose concentration |
| BloodPressure | Diastolic blood pressure (mm Hg) |
| SkinThickness | Triceps skin fold thickness (mm) |
| Insulin | 2-hour serum insulin (mu U/ml) |
| BMI | Body mass index |
| DiabetesPedigreeFunction | Genetic risk score based on family history |
| Age | Age in years |
| **Outcome** | **1 = diabetes, 0 = no diabetes (what we are predicting)** |

Source: National Institute of Diabetes and Digestive and Kidney Diseases  
Available on Kaggle: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

---
## Step 1: Import libraries

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)

from xgboost import XGBClassifier

---
## Step 2: Load and explore the dataset

In [4]:
# Load the dataset
df = pd.read_csv('diabetes.csv')

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
print(df.shape)

(768, 9)


In [6]:
print(df.columns)

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')


In [7]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [8]:
# How many patients have diabetes vs. no diabetes?
print(df['Outcome'].value_counts())
print('Diabetes prevalence:', df['Outcome'].mean() * 100,'%')

Outcome
0    500
1    268
Name: count, dtype: int64
Diabetes prevalence: 34.89583333333333 %


---
## Step 3: Split into features (X) and label (y)

- **X** = the 8 clinical measurements the model uses to make predictions
- **y** = the Outcome column — what we are trying to predict (1 = diabetes, 0 = no diabetes)

In [9]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

print('Features (X):', X.shape)
print('Labels (y):  ', y.shape)

Features (X): (768, 8)
Labels (y):   (768,)


---
## Step 4: Split into training and test sets

The model learns from the **training set** (80% of data).  
We evaluate it on the **test set** (20% of data) — patients the model has never seen before.

This is the same `train_test_split` approach from lecture.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training patients:', len(X_train))
print('Test patients:    ', len(X_test))

Training patients: 614
Test patients:     154


---
## Step 5: Train the XGBoost model

In [11]:
model = XGBClassifier(random_state=42)
model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


---
## Step 6: Make predictions

- `predict` gives a hard yes/no answer for each patient
- `predict_proba` gives a probability between 0 and 1 — this is the actual risk score, and we need it to calculate AUC

In [13]:
y_pred       = model.predict(X_test)               # yes/no prediction
y_pred_proba = model.predict_proba(X_test)[:, 1]   # risk probability (0 to 1)

print(y_pred_proba[:10])
print(y_test.values[:10])

[2.9575133e-01 4.7408052e-02 1.6202582e-01 1.6333961e-01 7.1539122e-01
 9.1557568e-01 1.3625469e-04 9.8648912e-01 6.7322510e-01 7.6502264e-01]
[0 0 0 0 0 0 0 0 0 0]


---
## Step 7: Evaluate the model

### Classification Report
Shows precision, recall, and F1-score for each class.

- **Precision** — of all patients flagged as diabetic, how many actually were?
- **Recall** — of all actual diabetic patients, how many did the model catch?
- **F1-score** — a balance between precision and recall

In healthcare, **recall matters most** — missing a real diabetes case is a more serious error than a false alarm.

### AUC-ROC Score
Measures how well the model separates diabetic from non-diabetic patients overall.
- **0.5** = no better than random guessing
- **1.0** = perfect
- **Above 0.80** is generally considered good for a medical prediction model

In [14]:
# Classification report
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

              precision    recall  f1-score   support

 No Diabetes       0.82      0.73      0.77        99
    Diabetes       0.59      0.71      0.64        55

    accuracy                           0.72       154
   macro avg       0.70      0.72      0.71       154
weighted avg       0.74      0.72      0.73       154



In [15]:
# AUC-ROC score
auc = roc_auc_score(y_test, y_pred_proba)
print('AUC-ROC Score:', auc)

AUC-ROC Score: 0.7761248852157944


---
## Step 8: Feature Importance

XGBoost tracks how often each feature was used across all trees. Features used more often get a higher importance score.

This connects to my capstone paper's theme of **explainability** — understanding why a model makes a prediction, not just whether it is accurate.

In [17]:
importances = model.feature_importances_
%matplotlib osx
plt.figure(figsize=(8, 5))
plt.bar(X.columns, importances, color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Feature Importance — Which clinical variables matter most?')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()


---
## Summary

In this notebook:
1. Loaded a real clinical dataset (Pima Indians Diabetes Dataset, 768 patients)
2. Split it into training and test sets using `train_test_split`
3. Trained an XGBoost classifier to predict diabetes onset
4. Evaluated the model using a classification report and AUC-ROC score
5. Visualized feature importance to understand which variables drive predictions

---
## Limitations and next steps

- This dataset only includes clinical variables, my full capstone model also incorporates lifestyle and neighborhood data
- The dataset is limited to one population group (Pima Indian women), which affects how broadly results apply
- Future work could compare XGBoost against other models (Logistic Regression, Random Forest) on a larger real-world dataset like NHANES